# 전처리

metadata 로드 - 전처리(시간,'Capacity','Re','Rct'수치형 변환) - 결측치 확인 - 시간 확인

In [1]:
# 도구 불러오기

import pandas as pd
import numpy as np
from collections import Counter
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import seaborn as sns
import platform
from glob import glob
import glob
import os
import re
import ast

# 판다스 출력 제한 해제 
pd.set_option('display.max_rows', 100) # 최대 100행까지 생략 없이 출력
pd.set_option('display.max_columns', None) 
pd.set_option('display.width', 1000)

In [3]:
# 원본 메타데이터 로드
df_meta = pd.read_csv("../../dataset/metadata.csv")

In [5]:
df_meta

,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct
0,discharge,[2010. 7. 21. 15. 0. ...,4,B0047,0,1,00001.csv,1.6743047446975208,NaN,NaN
1,impedance,[2010. 7. 21. 16. 53. ...,24,B0047,1,2,00002.csv,NaN,0.05605783343888099,0.20097016584458333
2,charge,[2010. 7. 21. 17. 25. ...,4,B0047,2,3,00003.csv,NaN,NaN,NaN
3,impedance,[2010 7 21 20 31 5],24,B0047,3,4,00004.csv,NaN,0.05319185850921101,0.16473399914864734
4,discharge,[2.0100e+03 7.0000e+00 2.1000e+01 2.1000e+01 2...,4,B0047,4,5,00005.csv,1.5243662105099023,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
7560,impedance,[2010. 9. 30. 7. 36. ...,24,B0055,247,7561,07561.csv,NaN,0.0968087979207628,0.15489738203707232
7561,discharge,[2010. 9. 30. 8. 8. ...,4,B0055,248,7562,07562.csv,1.0201379996149256,NaN,NaN
7562,charge,[2010. 9. 30. 8. 48. 54.25],4,B0055,249,7563,07563.csv,NaN,NaN,NaN
7563,discharge,[2010. 9. 30. 11. 50. ...,4,B0055,250,7564,07564.csv,0.9907591663373165,NaN,NaN


In [7]:
# 데이터 확인 (null/Dtype)
df_meta.info()

#-> 시간 object & Capacity,Re,Rct objct & Capacity,Re,Rct null 확인

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7565 entries, 0 to 7564
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   type                 7565 non-null   object
 1   start_time           7565 non-null   object
 2   ambient_temperature  7565 non-null   int64 
 3   battery_id           7565 non-null   object
 4   test_id              7565 non-null   int64 
 5   uid                  7565 non-null   int64 
 6   filename             7565 non-null   object
 7   Capacity             2794 non-null   object
 8   Re                   1956 non-null   object
 9   Rct                  1956 non-null   object
dtypes: int64(3), object(7)
memory usage: 591.1+ KB


In [4]:
df_meta['start_time'].describe()

count                                                  7565
unique                                                 2494
top       [2010.       9.      30.      12.      31.    ...
freq                                                      4
Name: start_time, dtype: object

In [8]:
# 시간 보정 실험 1 : 데이터 형식 분포 확인
# 1. 형식 분류 함수 정의
def detect_format(x):
    if pd.isna(x):
        return "NaN (결측치)"
    
    s = str(x).strip()
    if not s:
        return "Empty String (빈 문자열)"
    
    # 지수 표기법(e+03, E-01 등)이 포함되어 있는지 확인
    if re.search(r'[eE][+-]?\d+', s):
        return "Scientific Notation (지수 표기 포함: [2.01e+03 ...])"
    
    # 대괄호가 있고 일반 숫자로만 된 경우
    if s.startswith('[') and s.endswith(']'):
        return "Standard List (일반 리스트 형태: [2010. 9. ...])"
    
    # 그 외의 경우 (예상치 못한 노이즈)
    return f"Unexpected Format (기타 형식: {s[:20]}...)"

# 2. 형식별 케이스 카운트
format_counts = df_meta['start_time'].apply(detect_format).value_counts()

print("📊 [start_time] 데이터 형식 분포 확인")
print("-" * 50)
print(format_counts)
print("-" * 50)

# 3. 각 케이스별 실제 샘플 보기 (정확한 분석을 위해)
print("\n👀 형식별 실제 데이터 샘플 (첫 1개씩)")
for fmt in format_counts.index:
    sample = df_meta[df_meta['start_time'].apply(detect_format) == fmt]['start_time'].iloc[0]
    print(f"- {fmt}: {sample}")

📊 [start_time] 데이터 형식 분포 확인
--------------------------------------------------
start_time
Standard List (일반 리스트 형태: [2010. 9. ...])         6039
Scientific Notation (지수 표기 포함: [2.01e+03 ...])    1526
Name: count, dtype: int64
--------------------------------------------------

👀 형식별 실제 데이터 샘플 (첫 1개씩)
- Standard List (일반 리스트 형태: [2010. 9. ...]): [2010.       7.      21.      15.       0.      35.093]
- Scientific Notation (지수 표기 포함: [2.01e+03 ...]): [2.0100e+03 7.0000e+00 2.1000e+01 2.1000e+01 2.0000e+00 5.6984e+01]


In [9]:
# 시간 보정 실험 2 : 지수표기부 데이터패턴 확인
# 1. 지수 표기가 포함된 데이터만 필터링
sci_df = df_meta[df_meta['start_time'].str.contains('e', na=False, case=False)].copy()

def analyze_sub_pattern(x):
    s = str(x).strip()
    # 공백의 개수 (데이터 밀도 확인)
    space_count = len(re.findall(r'\s+', s))
    # 마침표의 개수
    dot_count = s.count('.')
    # 지수(e)의 개수 (연도만 지수인지, 월/일도 지수인지 확인)
    e_count = s.lower().count('e')
    
    return f"e={e_count}개, dot={dot_count}개, space={space_count}개"

# 2. 세부 패턴별 카운트
sub_pattern_counts = sci_df['start_time'].apply(analyze_sub_pattern).value_counts()

print(f"📊 Scientific Notation (총 {len(sci_df)}개) 내부 세부 패턴")
print("-" * 60)
print(sub_pattern_counts)
print("-" * 60)

# 3. 각 세부 패턴별 실제 샘플 2개씩 보기
print("\n👀 각 세부 패턴별 실제 데이터 샘플")
for pattern in sub_pattern_counts.index:
    samples = sci_df[sci_df['start_time'].apply(analyze_sub_pattern) == pattern]['start_time'].head(2).tolist()
    print(f"\n[ {pattern} ]")
    for i, s in enumerate(samples):
        print(f"  Sample {i+1}: {s}")

📊 Scientific Notation (총 1526개) 내부 세부 패턴
------------------------------------------------------------
start_time
e=6개, dot=6개, space=5개    1526
Name: count, dtype: int64
------------------------------------------------------------

👀 각 세부 패턴별 실제 데이터 샘플

[ e=6개, dot=6개, space=5개 ]
  Sample 1: [2.0100e+03 7.0000e+00 2.1000e+01 2.1000e+01 2.0000e+00 5.6984e+01]
  Sample 2: [2.010e+03 7.000e+00 2.200e+01 1.000e+00 4.000e+01 6.218e+00]


In [10]:
# 비정상 데이터 숫자확인 : 배터리id도 보면 좋긴하겟지만안함

def debug_raw_patterns(x):
    if pd.isna(x): return "Missing"
    try:
        # 1. [], e, E, +, - 등 문자열 전처리 (사용자님 제안 방식)
        clean_str = re.sub(r'[\[\]eE+]', ' ', str(x)) # 특수문자를 공백으로 치환
        
        # 2. 공백 기준으로 숫자 뭉치들 추출
        parts = clean_str.split()
        if len(parts) < 3: return f"Too Short (parts:{len(parts)})"
        
        # 3. 각 파트를 숫자로 변환 (지수 표기법의 소수점 처리를 위해 float -> int)
        nums = []
        for p in parts:
            try:
                # '2.0100' 같은 형태가 남아있을 수 있으므로 float 변환 후 반올림
                nums.append(int(round(float(p))))
            except: continue
            
        if len(nums) < 3: return "Number Conversion Failed"
        
        # 4. 연도 확인 (2008~2010 사이가 아니면 범인!)
        year = nums[0]
        if 2008 <= year <= 2010:
            return "Normal" # 정상 데이터는 패스
        else:
            return f"Year Out of Range: {year}"
            
    except Exception as e:
        return f"Error: {str(e)}"

# --- 실행 ---
df_debug = df_meta.copy()
df_debug['check_result'] = df_debug['start_time'].apply(debug_raw_patterns)

# 정상(Normal)이 아닌 것들만 추출
outliers = df_debug[df_debug['check_result'] != "Normal"]

print(f"🚩 이상 징후 데이터(비정상 연도 등): {len(outliers)}개")
print("-" * 100)
print(f"{'Battery_ID':<12} | {'Check Result':<25} | {'Original start_time'}")
print("-" * 100)

for _, row in outliers.head(50).iterrows():
    print(f"{str(row['battery_id']):<12} | {row['check_result']:<25} | {row['start_time']}")

🚩 이상 징후 데이터(비정상 연도 등): 1526개
----------------------------------------------------------------------------------------------------
Battery_ID   | Check Result              | Original start_time
----------------------------------------------------------------------------------------------------
B0047        | Year Out of Range: 2      | [2.0100e+03 7.0000e+00 2.1000e+01 2.1000e+01 2.0000e+00 5.6984e+01]
B0047        | Year Out of Range: 2      | [2.010e+03 7.000e+00 2.200e+01 1.000e+00 4.000e+01 6.218e+00]
B0047        | Year Out of Range: 2      | [2.0100e+03 7.0000e+00 2.4000e+01 2.0000e+00 2.3000e+01 2.1453e+01]
B0047        | Year Out of Range: 2      | [2.010e+03 7.000e+00 2.400e+01 1.200e+01 2.000e+00 2.839e+01]
B0047        | Year Out of Range: 2      | [2.0100e+03 7.0000e+00 2.8000e+01 2.0000e+00 2.9000e+01 3.7343e+01]
B0047        | Year Out of Range: 2      | [2.0100e+03 7.0000e+00 2.8000e+01 7.0000e+00 1.0000e+00 5.7687e+01]
B0047        | Year Out of Range: 2      | [2.010e+0

In [11]:
# 시간 실험 : 데이터 실패 46개의 값 확인

# [함수 1] 지수 표기법 대응 파싱 (보정/보간 없음)
def perfect_parse_time(x):
    if pd.isna(x): return pd.NaT
    try:
        s = str(x)
        # 지수 표기법 덩어리를 하나로 인식하는 정규식
        pattern = r"[-+]?\d*\.?\d+[eE][-+]?\d+|[-+]?\d+\.?\d*"
        parts = re.findall(pattern, s)
        
        if not parts: return pd.NaT
        
        # float 변환 후 정수화
        nums = [int(round(float(p))) for p in parts]
        
        if len(nums) >= 3:
            # 추출된 숫자 그대로 Timestamp 생성 시도
            # (만약 여기서 nums[0]이 2라면 0002년으로 생성됨 -> 에러 안남)
            # (단, 월이 0이거나 일이 32인 경우 등은 여기서 에러가 나서 NaT로 감)
            return pd.Timestamp(year=nums[0], month=nums[1], day=nums[2], 
                                hour=nums[3] if len(nums)>3 else 0,
                                minute=nums[4] if len(nums)>4 else 0,
                                second=nums[5] if len(nums)>5 else 0)
    except:
        pass
    return pd.NaT

# [함수 2] 수치형 데이터 정제
def clean_numeric(x):
    if pd.isna(x): return np.nan
    val = str(x).replace('[', '').replace(']', '').strip()
    try:
        return float(val)
    except:
        return np.nan

# --- [실행 및 검증 섹션] ---
df = df_meta.copy()

# 1. 파싱 적용
df['parsed_time'] = df['start_time'].apply(perfect_parse_time)

# 3. 결측치(NaT)인 데이터만 추출 ('문제가 되는 46개')
failed_cases = df[df['parsed_time'].isna()].copy()

print("-" * 60)
print(f"✅ 전체 데이터 중 파싱 실패(NaT) 개수: {len(failed_cases)}개")
print("-" * 60)

# 4. 실패한 데이터 46개의 원본 값 출력
if not failed_cases.empty:
    print(f"{'Index':<8} | {'Battery_ID':<12} | {'Original start_time'}")
    print("-" * 60)
    for idx, row in failed_cases.iterrows():
        print(f"{idx:<8} | {str(row['battery_id']):<12} | {row['start_time']}")
else:
    print("🎉 파싱에 실패한 데이터가 없습니다!")

# 5. (참고) 연도가 2000년 미만으로 나온 '이상한' 데이터가 있는지 추가 확인
# (에러는 안 났지만 연도가 잘못 찍힌 케이스 확인용)
weird_years = df[(df['parsed_time'].notna()) & (df['parsed_time'].dt.year < 2000)]
if not weird_years.empty:
    print(f"\n⚠️ 연도는 추출됐으나 2000년 미만인 데이터: {len(weird_years)}개")
    print(weird_years[['battery_id', 'start_time', 'parsed_time']].head())

------------------------------------------------------------
✅ 전체 데이터 중 파싱 실패(NaT) 개수: 46개
------------------------------------------------------------
Index    | Battery_ID   | Original start_time
------------------------------------------------------------
100      | B0047        | [2010.     8.     4.    20.    41.    59.5]
284      | B0045        | [2010.     8.     4.    20.    41.    59.5]
468      | B0048        | [2010.     8.     4.    20.    41.    59.5]
652      | B0046        | [2010.     8.     4.    20.    41.    59.5]
787      | B0043        | [2010.      6.      7.     14.     53.     59.75]
1026     | B0032        | [2009.       4.       9.      13.      33.      59.843]
1367     | B0029        | [2009.       4.       9.      13.      33.      59.843]
1478     | B0028        | [2.0090e+03 2.0000e+00 1.9000e+01 1.5000e+01 5.0000e+01 5.9703e+01]
1580     | B0042        | [2010.      6.      7.     14.     53.     59.75]
1957     | B0034        | [2.0090e+03 7.0000e+00 1.

In [21]:
# 위를 반영하여 전처리 (59.xx초에서 문제가 발생)

def clean_parse_time(x):
    if pd.isna(x):
        return pd.NaT
    
    try:
        s = str(x)
        pattern = r"[-+]?\d*\.?\d+[eE][-+]?\d+|[-+]?\d+\.?\d*"
        parts = re.findall(pattern, s)
        
        if len(parts) < 3:
            return pd.NaT
        
        nums = [float(p) for p in parts]
        
        year   = int(nums[0])
        month  = int(nums[1])
        day    = int(nums[2])
        hour   = int(nums[3]) if len(nums) > 3 else 0
        minute = int(nums[4]) if len(nums) > 4 else 0
        
        # 핵심: round 제거
        second = int(nums[5]) if len(nums) > 5 else 0
        
        return pd.Timestamp(year=year, month=month, day=day,
                            hour=hour, minute=minute, second=second)
    
    except Exception as e:
        print(f"[파싱 실패] {x} | {e}")
        return pd.NaT

def clean_numeric(x):
    """수치형 데이터의 대괄호 제거 및 float 변환"""
    if pd.isna(x): return np.nan
    val = str(x).replace('[', '').replace(']', '').strip()
    try:
        return float(val)
    except:
        return np.nan

# --- [전처리 실행] ---
df = df_meta.copy()

# 1. 시간 데이터 정제 (소수점 초까지 완벽 대응)
df['start_time'] = df['start_time'].apply(clean_parse_time)

# 2. 수치형 데이터 정제 (Capacity, Re, Rct)
for col in ['Capacity', 'Re', 'Rct']:
    if col in df.columns:
        df[col] = df[col].apply(clean_numeric)

# 3. 물리적 순서 정렬 (파일 번호 기준)
df['file_num'] = df['filename'].str.extract(r'(\d+)').astype(int)
df = df.sort_values(['battery_id', 'file_num']).reset_index(drop=True)

# 최종 결과 검증
print(f"전체 데이터 개수: {len(df):,}개")
print(f"시간 결측치(NaT): {df['start_time'].isna().sum()}개")
print(f"수치형 결측치(Cap): {df['Capacity'].isna().sum()}개")


전체 데이터 개수: 7,565개
시간 결측치(NaT): 0개
수치형 결측치(Cap): 4796개


In [13]:
df['start_time'].describe

<bound method NDFrame.describe of 0      2008-04-02 13:08:17
1      2008-04-02 15:25:41
2      2008-04-02 16:37:51
3      2008-04-02 19:43:48
4      2008-04-02 20:55:40
               ...        
7560   2010-09-30 07:36:45
7561   2010-09-30 08:08:36
7562   2010-09-30 08:48:54
7563   2010-09-30 11:50:17
7564   2010-09-30 12:31:10
Name: start_time, Length: 7565, dtype: datetime64[ns]>

In [14]:
df['start_time'].describe()

count                             7565
mean     2009-08-14 03:15:30.815994880
min                2008-04-02 13:08:17
25%                2008-07-11 18:25:27
50%                2009-08-02 09:10:15
75%                2010-06-30 09:21:28
max                2010-09-30 15:32:33
Name: start_time, dtype: object

In [ ]:
# 데이터 확인 (null/Dtype)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7565 entries, 0 to 7564
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   type                 7565 non-null   object        
 1   start_time           7565 non-null   datetime64[ns]
 2   ambient_temperature  7565 non-null   int64         
 3   battery_id           7565 non-null   object        
 4   test_id              7565 non-null   int64         
 5   uid                  7565 non-null   int64         
 6   filename             7565 non-null   object        
 7   Capacity             2769 non-null   float64       
 8   Re                   1947 non-null   float64       
 9   Rct                  1947 non-null   float64       
 10  file_num             7565 non-null   int64         
dtypes: datetime64[ns](1), float64(3), int64(4), object(3)
memory usage: 650.2+ KB


In [17]:
df.describe()

,start_time,ambient_temperature,test_id,uid,Capacity,Re,Rct,file_num
count,7565,7565.000000,7565.000000,7565.000000,2769.000000,1.947000e+03,1.947000e+03,7565.000000
mean,2009-08-14 03:15:30.815994880,20.017713,176.012558,3783.000000,1.326543,-4.976500e+11,1.055903e+12,3783.000000
min,2008-04-02 13:08:17,4.000000,0.000000,1.000000,0.000000,-9.689245e+14,-2.091081e+02,1.000000
25%,2008-07-11 18:25:27,4.000000,55.000000,1892.000000,1.150286,5.782157e-02,8.155754e-02,1892.000000
50%,2009-08-02 09:10:15,24.000000,129.000000,3783.000000,1.428065,7.255344e-02,1.014191e-01,3783.000000
75%,2010-06-30 09:21:28,24.000000,255.000000,5674.000000,1.673645,9.229960e-02,1.565123e-01,5674.000000
max,2010-09-30 15:32:33,44.000000,615.000000,7565.000000,2.640149,4.482291e+02,2.055843e+15,7565.000000
std,NaN,11.082914,152.174147,2183.971726,0.472517,2.195872e+13,4.659154e+13,2183.971726


In [15]:
df

,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct,file_num
0,charge,2008-04-02 13:08:17,24,B0005,0,5121,05121.csv,NaN,NaN,NaN,5121
1,discharge,2008-04-02 15:25:41,24,B0005,1,5122,05122.csv,1.856487,NaN,NaN,5122
2,charge,2008-04-02 16:37:51,24,B0005,2,5123,05123.csv,NaN,NaN,NaN,5123
3,discharge,2008-04-02 19:43:48,24,B0005,3,5124,05124.csv,1.846327,NaN,NaN,5124
4,charge,2008-04-02 20:55:40,24,B0005,4,5125,05125.csv,NaN,NaN,NaN,5125
...,...,...,...,...,...,...,...,...,...,...,...
7560,impedance,2010-09-30 07:36:45,24,B0056,247,7309,07309.csv,NaN,0.102677,0.170394,7309
7561,discharge,2010-09-30 08:08:36,4,B0056,248,7310,07310.csv,1.137273,NaN,NaN,7310
7562,charge,2010-09-30 08:48:54,4,B0056,249,7311,07311.csv,NaN,NaN,NaN,7311
7563,discharge,2010-09-30 11:50:17,4,B0056,250,7312,07312.csv,1.129059,NaN,NaN,7312


# 특이사항 확인 (결측치)

In [24]:
# 결측치 확인 (1.  discharge데이터에 Capacity의 결측치 여부 확인)
# 1. Discharge 데이터만 필터링
df_dis = df[df['type'] == 'discharge']

# 2. Capacity가 NaN인 행 추출
nan_capacity = df_dis[df_dis['Capacity'].isna()]

print(f"전체 Discharge 행 수: {len(df_dis):,}개")
print(f"Capacity가 NaN인 행 수: {len(nan_capacity):,}개")

if len(nan_capacity) > 0:
    print("결측치가 발견된 배터리 및 파일 목록:")
    # 어떤 파일에서 용량이 빠졌는지 요약 출력
    display(nan_capacity[['battery_id', 'filename']].drop_duplicates())
else:
    print("\n Discharge 데이터에 Capacity 값이 정상적으로 들어있습니다.")
# 확인시 25개의 결측치가 있었지만, 해당 배터리는 대상이 아님

전체 Discharge 행 수: 2,794개
Capacity가 NaN인 행 수: 25개
결측치가 발견된 배터리 및 파일 목록:


,battery_id,filename
6537,B0050,04371.csv
6539,B0050,04373.csv
6541,B0050,04375.csv
6543,B0050,04377.csv
6619,B0052,04391.csv
6623,B0052,04395.csv
6625,B0052,04397.csv
6627,B0052,04399.csv
6629,B0052,04401.csv
6631,B0052,04403.csv


In [25]:
# 결측치 확인 (2. impedance 데이터에 Re, RCT의 결측치 여부 확인)
# 1. Impedance 데이터만 필터링
df_imp = df[df['type'] == 'impedance']

# 2. Re 또는 Rct가 NaN인 행 추출
nan_imp = df_imp[df_imp['Re'].isna() | df_imp['Rct'].isna()]

print(f"전체 Impedance 행 수: {len(df_imp):,}개")
print(f"Re/Rct 결측 행 수: {len(nan_imp):,}개")

if len(nan_imp) > 0:
    print("\n결측치가 발견된 배터리 및 파일 목록 (상위 10개):")
    # 중복을 제거하고 파일 단위로 요약
    summary = nan_imp.groupby(['battery_id', 'filename']).size().reset_index(name='missing_rows')
    display(summary.head(10))
else:
    print("\n Impedance 데이터에 Re, Rct 값이 채워져 있습니다.")

# 3. (참고) 값의 범위 확인 (0이나 음수가 있는지 체크)
if len(df_imp) > len(nan_imp):
    print("\n--- [Re, Rct 값의 범위 요약] ---")
    display(df_imp[['Re', 'Rct']].describe())

# 결측이 있었으나, 해당 배터리는 대상이 아님

전체 Impedance 행 수: 1,956개
Re/Rct 결측 행 수: 9개

결측치가 발견된 배터리 및 파일 목록 (상위 10개):


,battery_id,filename,missing_rows
0,B0049,04268.csv,1
1,B0049,04280.csv,1
2,B0049,04282.csv,1
3,B0049,04292.csv,1
4,B0049,04294.csv,1
5,B0049,04306.csv,1
6,B0049,04316.csv,1
7,B0049,04318.csv,1
8,B0051,04454.csv,1



--- [Re, Rct 값의 범위 요약] ---


,Re,Rct
count,1.947000e+03,1.947000e+03
mean,-4.976500e+11,1.055903e+12
std,2.195872e+13,4.659154e+13
min,-9.689245e+14,-2.091081e+02
25%,5.782157e-02,8.155754e-02
50%,7.255344e-02,1.014191e-01
75%,9.229960e-02,1.565123e-01
max,4.482291e+02,2.055843e+15


In [26]:
# 데이터 시간 범위 확인
# 1. 전체 시간 범위 확인 (Min, Max)
start_date = df['start_time'].min()
end_date = df['start_time'].max()
duration = end_date - start_date

print("=== [데이터셋 시간 범위 결과] ===")
print(f"전체 시작 시점: {start_date}")
print(f"전체 종료 시점: {end_date}")
print(f"총 실험 기간  : {duration.days}일 {duration.seconds // 3600}시간")

# 2. 배터리별 시간 범위 확인 (각 배터리마다 실험 시점이 다를 수 있음)
print("\n=== [배터리 ID별 시간 범위] ===")
battery_time_range = df.groupby('battery_id')['start_time'].agg(['min', 'max', 'count', 'nunique'])
battery_time_range.columns = ['시작 시간', '종료 시간', '행 수', '파일 수']
display(battery_time_range)

# 전체 2008-04-02~2010-09-30 데이터였고,
# 해당 배터리는 2008-04-02~2008-05-28 (616사이클) & 07-07~08-20 (319사이클)이었음

=== [데이터셋 시간 범위 결과] ===
전체 시작 시점: 2008-04-02 13:08:17
전체 종료 시점: 2010-09-30 15:32:33
총 실험 기간  : 911일 2시간

=== [배터리 ID별 시간 범위] ===


,시작 시간,종료 시간,행 수,파일 수
battery_id,,,,
B0005,2008-04-02 13:08:17,2008-05-28 11:09:42,616,616
B0006,2008-04-02 13:08:17,2008-05-28 11:09:42,616,616
B0007,2008-04-02 13:08:17,2008-05-28 11:09:42,616,616
B0018,2008-07-07 12:26:45,2008-08-20 08:37:19,319,319
B0025,2009-02-13 19:03:52,2009-03-19 12:55:26,80,80
B0026,2009-02-13 19:03:52,2009-03-19 12:55:26,80,80
B0027,2009-02-13 19:03:52,2009-03-19 12:55:26,80,80
B0028,2009-02-13 19:03:52,2009-03-19 12:55:26,80,80
B0029,2009-04-07 15:59:18,2009-04-18 01:29:13,97,97


In [31]:
# 'B0005', 'B0006', 'B0007', 'B0018' 배터리 대상으로 충전 / 방전 / 임피던스별 data 구성하기

# 대상 배터리 ID 리스트
target_battery_ids = ['B0005', 'B0006', 'B0007', 'B0018']

# 데이터 저장할 폴더 경로
data_folder = "../../dataset/data"

# 데이터를 쌓아두기 위한 저장소(Dictionary) 생성
collected_data = {}


### 각 배터리별 EOL 사이클 번호를 먼저 파악 (RUL 계산용)
# SOH 80% 기준을 찾기 위해 메타데이터(df)에서 미리 계산
eol_dict = {}
for b_id in target_battery_ids:
    b_meta = df[(df['battery_id'] == b_id) & (df['type'] == 'discharge')].copy()
    if not b_meta.empty:
        initial_cap = b_meta['Capacity'].iloc[0] # 첫 번째 방전 용량
        # SOH 80% 이하인 첫 번째 행의 인덱스(순번) 찾기
        eol_idx = np.where((b_meta['Capacity'] / initial_cap) * 100 <= 80)[0]
        eol_dict[b_id] = eol_idx[0] + 1 if len(eol_idx) > 0 else np.nan

for battery_id in target_battery_ids:
    # 시간순 정렬 (매칭 오류 방지)
    filtered_df = df[df['battery_id'] == battery_id].sort_values('start_time')
    cycle_counters = {'charge': 1, 'discharge': 1, 'impedance': 1}
    
    # 기준 용량 정의 (해당 배터리의 첫 번째 방전 용량)
    dis_rows = filtered_df[filtered_df['type'] == 'discharge']
    first_cap = dis_rows['Capacity'].iloc[0] if not dis_rows.empty else None

    for _, row in filtered_df.iterrows():
        file_path = os.path.join(data_folder, row['filename'])

        if os.path.exists(file_path):
            temp_df = pd.read_csv(file_path)
            d_type = row['type']
            current_cycle = cycle_counters[d_type]

            # --- [1단계: 파생 변수 로직 정의] ---
            # SOH 정의: 오직 discharge 타입일 때만 해당 사이클의 용량으로 계산
            if d_type == 'discharge' and first_cap and pd.notnull(row['Capacity']):
                soh_val = (row['Capacity'] / first_cap) * 100
            else:
                soh_val = np.nan # 다른 타입은 우선 NaN 처리
            
            # EOL 정의: 미리 계산된 eol_dict 이용
            eol_val = eol_dict.get(battery_id)
            
            # RUL 정의: (사망 사이클 - 현재 사이클)
            rul_val = (eol_val - current_cycle) if pd.notnull(eol_val) else np.nan
            if pd.notnull(rul_val): rul_val = max(0, rul_val)

            # --- [2단계: 데이터프레임 열 추가] ---
            temp_df['start_time'] = row['start_time']
            temp_df['battery_id'] = battery_id
            temp_df['type'] = d_type
            temp_df['ambient_temperature'] = row['ambient_temperature']
            
            temp_df['cycle'] = current_cycle
            temp_df['SOH'] = soh_val  # 정의된 값 주입
            temp_df['EOL_cycle'] = eol_val
            temp_df['RUL'] = rul_val

            # 타입별 특화 데이터 추가
            if d_type == 'discharge':
                temp_df['Capacity'] = row['Capacity']
            elif d_type == 'impedance':
                temp_df['Re'] = row['Re']
                temp_df['Rct'] = row['Rct']

            # 카운터 증가 및 저장
            cycle_counters[d_type] += 1
            key = f"df_{d_type}_{battery_id}"
            if key not in collected_data:
                collected_data[key] = []
            collected_data[key].append(temp_df)

# --- [3단계: 데이터 통합 및 결측치 전파] ---
for key, df_list in collected_data.items():
    combined_df = pd.concat(df_list, ignore_index=True)
    
    # SOH 전파: discharge 파일들 사이의 간극(charge, impedance 등)을 메워줌
    # 단, 사용자님이 'discharge에만' 있길 원하신다면 이 섹션을 제외하면 됩니다.
    # 만약 모든 행에 SOH가 필요하다면 아래 ffill/bfill을 유지하세요.
    if 'SOH' in combined_df.columns:
        combined_df['SOH'] = combined_df['SOH'].ffill()
        combined_df['SOH'] = combined_df['SOH'].bfill()
    
    globals()[key] = combined_df
    print(f"✅ {key} 생성 완료 | 파일 {len(df_list)}개 통합 | 크기: {combined_df.shape} | SOH 결측치: {combined_df['SOH'].isnull().sum()} | cycle/SOH/EOL/RUL 파생 완료")


✅ df_charge_B0005 생성 완료 | 파일 170개 통합 | 크기: (541173, 14) | SOH 결측치: 541173 | cycle/SOH/EOL/RUL 파생 완료
✅ df_discharge_B0005 생성 완료 | 파일 168개 통합 | 크기: (50285, 15) | SOH 결측치: 0 | cycle/SOH/EOL/RUL 파생 완료
✅ df_impedance_B0005 생성 완료 | 파일 278개 통합 | 크기: (13344, 15) | SOH 결측치: 13344 | cycle/SOH/EOL/RUL 파생 완료
✅ df_charge_B0006 생성 완료 | 파일 170개 통합 | 크기: (541173, 14) | SOH 결측치: 541173 | cycle/SOH/EOL/RUL 파생 완료
✅ df_discharge_B0006 생성 완료 | 파일 168개 통합 | 크기: (50285, 15) | SOH 결측치: 0 | cycle/SOH/EOL/RUL 파생 완료
✅ df_impedance_B0006 생성 완료 | 파일 278개 통합 | 크기: (13344, 15) | SOH 결측치: 13344 | cycle/SOH/EOL/RUL 파생 완료
✅ df_charge_B0007 생성 완료 | 파일 170개 통합 | 크기: (541173, 14) | SOH 결측치: 541173 | cycle/SOH/EOL/RUL 파생 완료
✅ df_discharge_B0007 생성 완료 | 파일 168개 통합 | 크기: (50285, 15) | SOH 결측치: 0 | cycle/SOH/EOL/RUL 파생 완료
✅ df_impedance_B0007 생성 완료 | 파일 278개 통합 | 크기: (13344, 15) | SOH 결측치: 13344 | cycle/SOH/EOL/RUL 파생 완료
✅ df_charge_B0018 생성 완료 | 파일 134개 통합 | 크기: (279810, 14) | SOH 결측치: 279810 | cycle/SOH/EOL/RUL 파생 완료
✅ df_i

In [35]:
df_discharge_B0005.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50285 entries, 0 to 50284
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Voltage_measured      50285 non-null  float64       
 1   Current_measured      50285 non-null  float64       
 2   Temperature_measured  50285 non-null  float64       
 3   Current_load          50285 non-null  float64       
 4   Voltage_load          50285 non-null  float64       
 5   Time                  50285 non-null  float64       
 6   start_time            50285 non-null  datetime64[ns]
 7   battery_id            50285 non-null  object        
 8   type                  50285 non-null  object        
 9   ambient_temperature   50285 non-null  int64         
 10  cycle                 50285 non-null  int64         
 11  SOH                   50285 non-null  float64       
 12  EOL_cycle             50285 non-null  int64         
 13  RUL             

In [44]:
df_discharge_B0005

,Voltage_measured,Current_measured,Temperature_measured,Current_load,Voltage_load,Time,start_time,battery_id,type,ambient_temperature,cycle,SOH,EOL_cycle,RUL,Capacity
0,4.191492,-0.004902,24.330034,-0.0006,0.000,0.000,2008-04-02 15:25:41,B0005,discharge,24,1,100.000000,101,100,1.856487
1,4.190749,-0.001478,24.325993,-0.0006,4.206,16.781,2008-04-02 15:25:41,B0005,discharge,24,1,100.000000,101,100,1.856487
2,3.974871,-2.012528,24.389085,-1.9982,3.062,35.703,2008-04-02 15:25:41,B0005,discharge,24,1,100.000000,101,100,1.856487
3,3.951717,-2.013979,24.544752,-1.9982,3.030,53.781,2008-04-02 15:25:41,B0005,discharge,24,1,100.000000,101,100,1.856487
4,3.934352,-2.011144,24.731385,-1.9982,3.011,71.922,2008-04-02 15:25:41,B0005,discharge,24,1,100.000000,101,100,1.856487
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50280,3.579262,-0.001569,34.864823,0.0006,0.000,2781.312,2008-05-27 20:45:42,B0005,discharge,24,168,71.375616,101,0,1.325079
50281,3.581964,-0.003067,34.814770,0.0006,0.000,2791.062,2008-05-27 20:45:42,B0005,discharge,24,168,71.375616,101,0,1.325079
50282,3.584484,-0.003079,34.676258,0.0006,0.000,2800.828,2008-05-27 20:45:42,B0005,discharge,24,168,71.375616,101,0,1.325079
50283,3.587336,0.001219,34.565580,0.0006,0.000,2810.640,2008-05-27 20:45:42,B0005,discharge,24,168,71.375616,101,0,1.325079


In [34]:
# 간단 확인
display(df_charge_B0006)
display(df_discharge_B0007)
display(df_impedance_B0005) 

,Voltage_measured,Current_measured,Temperature_measured,Current_charge,Voltage_charge,Time,start_time,battery_id,type,ambient_temperature,cycle,SOH,EOL_cycle,RUL
0,3.864624,0.000082,24.682214,-0.001,-0.007,0.000,2008-04-02 13:08:17,B0006,charge,24,1,NaN,61,60
1,3.469113,-4.059185,24.695407,-4.060,1.558,2.532,2008-04-02 13:08:17,B0006,charge,24,1,NaN,61,60
2,3.994806,1.513750,24.711491,1.506,4.710,5.500,2008-04-02 13:08:17,B0006,charge,24,1,NaN,61,60
3,4.005888,1.511389,24.739672,1.506,4.726,8.344,2008-04-02 13:08:17,B0006,charge,24,1,NaN,61,60
4,4.012944,1.510817,24.753180,1.506,4.737,11.125,2008-04-02 13:08:17,B0006,charge,24,1,NaN,61,60
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
541168,0.979650,-0.005157,22.984433,0.000,-0.007,0.000,2008-05-28 11:09:42,B0006,charge,24,170,NaN,61,0
541169,-0.001422,-0.001326,22.984099,-0.001,-0.007,2.547,2008-05-28 11:09:42,B0006,charge,24,170,NaN,61,0
541170,4.985089,0.000203,22.986148,0.000,4.996,5.500,2008-05-28 11:09:42,B0006,charge,24,170,NaN,61,0
541171,4.985345,0.000329,22.994625,-0.001,4.996,8.312,2008-05-28 11:09:42,B0006,charge,24,170,NaN,61,0


,Voltage_measured,Current_measured,Temperature_measured,Current_load,Voltage_load,Time,start_time,battery_id,type,ambient_temperature,cycle,SOH,EOL_cycle,RUL,Capacity
0,4.199360,-0.001866,23.937044,-0.0004,0.000,0.000,2008-04-02 15:25:41,B0007,discharge,24,1,100.000000,124,123,1.891052
1,4.199497,-0.002139,23.924074,-0.0004,4.215,16.781,2008-04-02 15:25:41,B0007,discharge,24,1,100.000000,124,123,1.891052
2,3.985606,-1.988778,24.004257,-2.0000,3.003,35.703,2008-04-02 15:25:41,B0007,discharge,24,1,100.000000,124,123,1.891052
3,3.963247,-1.992558,24.162868,-2.0000,2.987,53.781,2008-04-02 15:25:41,B0007,discharge,24,1,100.000000,124,123,1.891052
4,3.946647,-1.988491,24.346368,-2.0000,2.972,71.922,2008-04-02 15:25:41,B0007,discharge,24,1,100.000000,124,123,1.891052
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50280,3.336677,-0.002464,38.744012,0.0006,0.001,2781.312,2008-05-27 20:45:42,B0007,discharge,24,168,75.749109,124,0,1.432455
50281,3.349952,-0.005358,38.462399,0.0006,0.001,2791.062,2008-05-27 20:45:42,B0007,discharge,24,168,75.749109,124,0,1.432455
50282,3.362104,-0.003906,38.246805,0.0006,0.001,2800.828,2008-05-27 20:45:42,B0007,discharge,24,168,75.749109,124,0,1.432455
50283,3.373357,-0.002763,37.970504,0.0006,0.001,2810.640,2008-05-27 20:45:42,B0007,discharge,24,168,75.749109,124,0,1.432455


,Sense_current,Battery_current,Current_ratio,Battery_impedance,Rectified_Impedance,start_time,battery_id,type,ambient_temperature,cycle,SOH,EOL_cycle,RUL,Re,Rct
0,(-1+1j),(-1+1j),(1+0j),(-0.43892624830326377-0.107298295835479j),(0.07006937798290404-0.00047998469078178944j),2008-04-18 20:55:29,B0005,impedance,24,1,NaN,101,100,0.044669,0.069456
1,(820.6094970703125-36.23455047607422j),(337.0914611816406-82.9207763671875j),(2.3204145178633437+0.4633045948164565j),(0.13008840651776496-0.19711481029612374j),(0.06817886114940203-0.001190040925296937j),2008-04-18 20:55:29,B0005,impedance,24,1,NaN,101,100,0.044669,0.069456
2,(827.2421875-48.23122787475586j),(330.6315612792969-70.01371765136719j),(2.424192647592199+0.36746495469515333j),(0.058770560504133235+0.03330656583655633j),(0.06793257733714593-5.6826811936507056e-05j),2008-04-18 20:55:29,B0005,impedance,24,1,NaN,101,100,0.044669,0.069456
3,(827.1934814453125-56.195716857910156j),(330.8086242675781-61.73442459106445j),(2.4470021712116985+0.28677775364826635j),(0.0058135116366746726-0.060546548141956195j),(0.06691839226387165-0.0008787264015490232j),2008-04-18 20:55:29,B0005,impedance,24,1,NaN,101,100,0.044669,0.069456
4,(824.9295043945312-53.241477966308594j),(332.68267822265625-57.62901306152344j),(2.434304977711638+0.2616460702282485j),(0.12608106668700975-0.09044390544679616j),(0.06807105294348659-0.0001974802021297548j),2008-04-18 20:55:29,B0005,impedance,24,1,NaN,101,100,0.044669,0.069456
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13339,(915.489013671875-64.04512023925781j),(230.14950561523438+91.9098892211914j),(3.334834919457856-1.6100379067382284j),(0.2450237439418051-0.04983619374573483j),NaN,2008-05-27 21:34:28,B0005,impedance,24,278,NaN,101,0,0.050036,0.074792
13340,(916.7255249023438+2.986217498779297j),(212.18885803222656+107.74581146240234j),(3.4403927232922893-1.732898190940107j),(0.26459423551817796-0.055235088103719486j),NaN,2008-05-27 21:34:28,B0005,impedance,24,278,NaN,101,0,0.050036,0.074792
13341,(914.61962890625+126.11148071289062j),(176.59803771972656+131.6827850341797j),(3.6706560245778346-2.0229597798418966j),(0.28857059933181783-0.058837097064953735j),NaN,2008-05-27 21:34:28,B0005,impedance,24,278,NaN,101,0,0.050036,0.074792
13342,(880.3408203125+293.8252868652344j),(136.84762573242188+146.8810272216797j),(4.060163697383427-2.2107488242884408j),(0.3176999700278338-0.05912720702491042j),NaN,2008-05-27 21:34:28,B0005,impedance,24,278,NaN,101,0,0.050036,0.074792


In [ ]:
# 분석가 팁: 반복되는 작업을 줄이기 위해 리스트를 순회하며 출력합니다.

# 1. 방전(Discharge) 데이터 확인
print("=== [DISCHARGE DATA SHAPE & PREVIEW] ===")
discharge_dfs = [df_discharge_B0005, df_discharge_B0006, df_discharge_B0007, df_discharge_B0018]
df_names_dis = ['B0005', 'B0006', 'B0007', 'B0018']

for name, df_obj in zip(df_names_dis, discharge_dfs):
    print(f"Discharge_{name} Shape: {df_obj.shape}")
    display(df_obj.head())  # 전체 출력 대신 head()를 권장하지만, 원하시면 그대로 사용 가능합니다.
    print("-" * 50)

# 2. 임피던스(Impedance) 데이터 확인
print("\n=== [IMPEDANCE DATA SHAPE & PREVIEW] ===")
impedance_dfs = [df_impedance_B0005, df_impedance_B0006, df_impedance_B0007, df_impedance_B0018]
df_names_imp = ['B0005', 'B0006', 'B0007', 'B0018']

for name, df_obj in zip(df_names_imp, impedance_dfs):
    print(f"Impedance_{name} Shape: {df_obj.shape}")
    display(df_obj.head())
    print("-" * 50)

# 3. 충전(Charge) 데이터 확인
print("\n=== [CHARGE DATA SHAPE & PREVIEW] ===")
charge_dfs = [df_charge_B0005, df_charge_B0006, df_charge_B0007, df_charge_B0018]
df_names_cha = ['B0005', 'B0006', 'B0007', 'B0018']

for name, df_obj in zip(df_names_cha, charge_dfs):
    print(f"Charge_{name} Shape: {df_obj.shape}")
    display(df_obj.head())
    print("-" * 50)